# Chapter 8: a soft sensor for final RDF, the VDK forecast and the DO flag
300 synthetic lager fills, seed 42. The oldest 225 train, the newest 75 test. A linear model and gradient boosting (scikit-learn) against the training average.

**No code tier.** In a spreadsheet, use TREND to fit final RDF on the 48 hour AE for the 225 training rows and predict the 75 test rows. Subtract the lab result. Work the mean and sd of the differences and draw mean plus and minus 1.96 sd.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
f = pd.read_csv(DATA + "ch08_fills.csv")
feats = ["ae_48h_plato", "og_plato", "mean_temp_c", "pitch_mcells_ml", "yeast_generation"]
train, test = f[f.split == "train"], f[f.split == "test"]   # a time split: never shuffle
lin = LinearRegression().fit(train[feats], train.rdf_pct)
gb = GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=42).fit(train[feats], train.rdf_pct)
base = train.rdf_pct.mean()
rows = []
for name, pred in (("baseline (training average)", np.full(len(test), base)), ("linear model", lin.predict(test[feats])), ("gradient boosting", gb.predict(test[feats]))):
    d = pred - test.rdf_pct.to_numpy(); b, s = d.mean(), d.std(ddof=1)
    rows.append(dict(model=name, MAE=round(np.abs(d).mean(), 2), bias=round(b, 2), sd=round(s, 2), LoA_low=round(b - 1.96 * s, 2), LoA_high=round(b + 1.96 * s, 2)))
print("training average", round(base, 1)); pd.DataFrame(rows)

In [ ]:
p = lin.predict(test[feats]); y = test.rdf_pct.to_numpy(); d = p - y
plt.figure(figsize=(5, 5)); plt.scatter(y, p, s=12, color="#2f7f72")
lo, hi = 64, 72; xs = np.array([lo, hi])
plt.plot(xs, xs, color="#0b4f45"); plt.fill_between(xs, xs + d.mean() - 1.96 * d.std(ddof=1), xs + d.mean() + 1.96 * d.std(ddof=1), color="#cfe6e1", alpha=0.6)
plt.xlabel("measured RDF (lab)"); plt.ylabel("predicted RDF at 48 hours"); plt.xlim(lo, hi); plt.ylim(lo, hi); plt.show()
print("model only spread:", round(math.sqrt(0.67 ** 2 - 0.34 ** 2), 2))

## Diacetyl: forecast the end of maturation (8.3)

In [ ]:
vdk = {3: 0.31, 4: 0.19, 5: 0.12}
q = np.mean([vdk[4] / vdk[3], vdk[5] / vdk[4]])
print("mean ratio", round(q, 2), "| day 6", round(0.12 * 0.62, 3), "| day 7", round(0.12 * 0.62 * 0.62, 3))

## DO at the filler: flag from the run's own history (8.4)

In [ ]:
do = pd.read_csv(DATA + "ch08_do_filler_run.csv")
b = do.do_ppb[:8]; flag = b.mean() + 3 * b.std(ddof=1)
first = do[(do.reading > 8) & (do.do_ppb > flag)].reading.min()
print(f"baseline mean {b.mean():.2f} sd {b.std(ddof=1):.1f} flag line {flag:.1f} ppb; first flag at reading {first}")
plt.plot(do.reading, do.do_ppb, "-o", color="#2f7f72"); plt.axhline(flag, ls="--", color="#c27c0e"); plt.axhline(50, color="#b23a2e")
plt.xlabel("reading (hourly)"); plt.ylabel("DO at the filler feed (ppb)"); plt.show()

**What to look at.** Both models halve the baseline error and gradient boosting does no better than the line. Judge the model on its limits of agreement, not a correlation. Azure Machine Learning designer or Excel can build the linear version.